# MeterIQ Data Preprocessing & Exploratory Data Analysis (EDA)
This notebook implements the complete data preprocessing, feature engineering, and detailed exploratory analysis pipeline for the MeterIQ project.

## 1. Project/Data Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import json
from preprocessing.cleaning import standardize_dataset, clean_dataset
from preprocessing.validation import validate_preprocessing
from preprocessing.feature_engineering import (
    add_time_features, add_cyclical_features, add_lag_features, 
    add_rolling_features, add_anomaly_features, add_forecasting_features
)
from preprocessing.eda import (
    plot_consumption_distribution, plot_temporal_patterns, 
    plot_meter_comparison, plot_feature_correlations, plot_gap_distribution
)

DATA_DIR = 'Desktop/MeterIQ/data/'
PROCESSED_DIR = 'Desktop/MeterIQ/data/processed/'
FEATURES_DIR = 'Desktop/MeterIQ/data/features/'
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)

## 2. Automatic Dataset Discovery

In [ ]:
all_files = glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True)
high_res_files = [f for f in all_files if 'Aggregated' not in f]
print(f"Discovered {len(high_res_files)} high-resolution datasets.")
for f in high_res_files:
    print(f"- {os.path.basename(f)}")

## 3. Dataset Inspection

In [ ]:
for f in high_res_files[:2]: # Inspect first 2 for brevity
    df = pd.read_csv(f, nrows=5)
    print(f"\nFile: {os.path.basename(f)}")
    print(df.head())

## 4. Data Quality Analysis

In [ ]:
quality_report = {}
for f in high_res_files:
    df = pd.read_csv(f)
    cols = df.columns.tolist()
    ts_col = 'x_Timestamp' if 'x_Timestamp' in cols else 'Date'
    cons_col = 't_kWh'
    meter_col = 'meter'
    
    quality_report[os.path.basename(f)] = {
        'total_rows': len(df),
        'missing': df.isnull().sum().to_dict(),
        'duplicates': df.duplicated().sum(),
        'mt_duplicates': df.duplicated(subset=[meter_col, ts_col]).sum() if meter_col in cols and ts_col in cols else 'N/A',
        'negative_cons': (df[cons_col] < 0).sum() if cons_col in cols else 'N/A'
    }
print(pd.DataFrame(quality_report).T)

## 5. Cleaning and Standardization

In [ ]:
processed_files = []
for f in high_res_files:
    df = pd.read_csv(f)
    df_std = standardize_dataset(df)
    df_clean = clean_dataset(df_std)
    
    out_name = f"processed_{os.path.basename(f)}"
    out_path = os.path.join(PROCESSED_DIR, out_name)
    df_clean.to_csv(out_path, index=False)
    processed_files.append(out_path)
    print(f"Processed {os.path.basename(f)}")

## 6. Timestamp and Sampling Analysis

In [ ]:
sampling_results = []
for f in processed_files:
    df = pd.read_csv(f)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Calculate intervals for first meter
    meter_id = df['meter_id'].iloc[0]
    meter_df = df[df['meter_id'] == meter_id].sort_values('timestamp')
    diffs = meter_df['timestamp'].diff().dt.total_seconds()
    
    sampling_results.append({
        'file': os.path.basename(f),
        'median_interval': diffs.median(),
        'mode_interval': diffs.mode()[0] if not diffs.mode().empty else None
    })
print(pd.DataFrame(sampling_results))

## 7. Feature Engineering

In [ ]:
final_datasets = []
for f in processed_files:
    df = pd.read_csv(f)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    df = add_time_features(df)
    df = add_cyclical_features(df)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_anomaly_features(df)
    df = add_forecasting_features(df)
    
    out_name = f"features_{os.path.basename(f)}"
    out_path = os.path.join(FEATURES_DIR, out_name)
    df.to_csv(out_path, index=False)
    final_datasets.append(out_path)
    print(f"Features generated for {os.path.basename(f)}")

## 8. Comprehensive Exploratory Data Analysis (EDA)

In [ ]:
sample_df = pd.read_csv(final_datasets[0])
sample_df['timestamp'] = pd.to_datetime(sample_df['timestamp'])

print("\n--- Consumption Distribution ---")
plot_consumption_distribution(sample_df)

print("\n--- Temporal Patterns (Hourly & Daily) ---")
plot_temporal_patterns(sample_df)

print("\n--- Random Meter Comparisons ---")
plot_meter_comparison(sample_df)

print("\n--- Feature Correlation Analysis ---")
plot_feature_correlations(sample_df)

print("\n--- Sampling Gap Distribution ---")
plot_gap_distribution(sample_df)

## 9. ML-Ready Dataset Generation & Validation

In [ ]:
validation_results = []
for f in final_datasets:
    df = pd.read_csv(f)
    is_valid, errors = validate_preprocessing(df)
    validation_results.append({
        'file': os.path.basename(f),
        'valid': is_valid,
        'errors': errors
    })
print(pd.DataFrame(validation_results))

## 10. Summary Table

In [ ]:
summary_data = []
for f_feat in final_datasets:
    df = pd.read_csv(f_feat)
    summary_data.append({
        'Dataset': os.path.basename(f_feat),
        'Rows': len(df),
        'Meters': df['meter_id'].nunique(),
        'Features': len(df.columns)
    })
print(pd.DataFrame(summary_data))